# Workstation Rental Price Tier Prediction - Improved Pipeline

**Goal:** Push macro-F1 from 0.69 → 0.70+

**Key improvements over baseline:**
1. Full feature engineering (facilities, geo, group aggs, interactions, ratios)
2. LightGBM + XGBoost + CatBoost ensemble via stacking
3. Better hyperparameter tuning with Optuna
4. Proper early stopping within CV
5. Threshold optimization for macro-F1

In [3]:
import json, os, time, warnings, collections
import numpy as np, pandas as pd
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, Pool
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder, StandardScaler, MultiLabelBinarizer
from sklearn.model_selection import GroupKFold, StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

OUT = "/kaggle/working"
os.makedirs(OUT, exist_ok=True)
print("Setup complete")

Setup complete


## 1. Load & Clean Data

In [4]:
D = "/kaggle/input/competitions/workstation-rental-price-tier-prediction/kaggle_ready_dataset/"
train_raw = pd.read_csv(D + "train.csv")
test_raw = pd.read_csv(D + "test.csv")
sample_sub = pd.read_csv(D + "sample_submission.csv")

print("Raw shapes:", train_raw.shape, test_raw.shape)
print("Class balance:\n", train_raw.price_tier.value_counts(normalize=True).sort_index().round(3))

# Clean: remove contradicting duplicates
feat_cols = [c for c in train_raw.columns if c not in ("space_id", "price_tier")]
dup_mask = train_raw.duplicated(subset=feat_cols, keep=False)
dd = train_raw[dup_mask]
bad_idx = dd.index[dd.groupby(feat_cols, dropna=False)["price_tier"].transform("nunique") > 1]
train_clean = train_raw.drop(index=bad_idx).drop_duplicates(subset=feat_cols).reset_index(drop=True)
print(f"Cleaned: {len(train_raw)} -> {len(train_clean)} rows")

Raw shapes: (279746, 60) (114538, 59)
Class balance:
 price_tier
1    0.239
2    0.291
3    0.217
4    0.252
Name: proportion, dtype: float64
Cleaned: 279746 -> 278543 rows


## 2. Comprehensive Feature Engineering

In [5]:
def build_features(df, all_data=None, fit_mode=True, memo=None):
    """
    Comprehensive feature engineering.
    all_data: combined train+test for computing global statistics (no target leakage)
    fit_mode: True for train, computes and stores aggregates in memo dict
    memo: dict to store/retrieve fitted aggregates for test consistency
    """
    d = df.copy()
    if memo is None:
        memo = {}
    
    # ---- Date features ----
    for c in ["snapshot_start_date", "snapshot_end_date", "first_review_date", "last_review_date"]:
        d[c] = pd.to_datetime(d[c], errors="coerce")
    
    d["snapshot_span_days"] = (d["snapshot_end_date"] - d["snapshot_start_date"]).dt.days
    d["days_since_first_review"] = (d["snapshot_end_date"] - d["first_review_date"]).dt.days
    d["days_since_last_review"] = (d["snapshot_end_date"] - d["last_review_date"]).dt.days
    d["snapshot_month"] = d["snapshot_start_date"].dt.month
    d["snapshot_year"] = d["snapshot_start_date"].dt.year
    d["review_recency_ratio"] = d["days_since_last_review"] / d["days_since_first_review"].replace(0, np.nan)
    d["review_span_days"] = d["days_since_first_review"] - d["days_since_last_review"]
    
    # ---- Tenure features ----
    d["platform_tenure"] = d["operator_platform_age_years"] + d["operator_platform_age_months"] / 12
    d["active_tenure"] = d["operator_active_years"] + d["operator_active_months"] / 12
    d["inactive_gap"] = d["platform_tenure"] - d["active_tenure"]
    d["activity_ratio"] = d["active_tenure"] / d["platform_tenure"].replace(0, np.nan)
    
    # ---- Review / Score features ----
    score_cols = [c for c in d.columns if c.startswith("score_")]
    d["has_reviews"] = d["score_overall"].notna().astype(int)
    d["score_mean"] = d[score_cols].mean(axis=1)
    d["score_std"] = d[score_cols].std(axis=1)
    d["score_min"] = d[score_cols].min(axis=1)
    d["score_max"] = d[score_cols].max(axis=1)
    d["score_range"] = d["score_max"] - d["score_min"]
    d["recent_review_share"] = d["review_count_12m"] / d["review_count"].replace(0, np.nan)
    d["reviews_per_year"] = d["review_count"] / d["platform_tenure"].replace(0, np.nan)
    
    # ---- Operator portfolio features ----
    u = d["operator_metro_units"].replace(0, np.nan)
    d["share_whole"] = d["operator_metro_whole"] / u
    d["share_private"] = d["operator_metro_private"] / u
    d["share_shared"] = d["operator_metro_shared"] / u
    d["metro_share"] = d["operator_metro_units"] / d["operator_portfolio_size"].replace(0, np.nan)
    d["metro_concentration"] = d["operator_metro_units"] / d["operator_portfolio_size"].replace(0, np.nan)
    
    # ---- Space features ----
    d["seats_per_ws"] = d["max_seats"] / d["workstations"].replace(0, np.nan)
    d["rooms_per_ws"] = d["utility_rooms"] / d["workstations"].replace(0, np.nan)
    d["seats_per_room"] = d["max_seats"] / d["utility_rooms"].replace(0, np.nan)
    
    # ---- Data quality flags ----
    d["metro_sum_mismatch"] = (d["operator_metro_units"] !=
        d["operator_metro_whole"] + d["operator_metro_private"] + d["operator_metro_shared"]).astype(int)
    d["metro_over_portfolio"] = (d["operator_metro_units"] > d["operator_portfolio_size"]).astype(int)
    d["active_gt_platform"] = (d["operator_active_years"] > d["operator_platform_age_years"]).astype(int)
    
    # ---- Facilities features ----
    fac = d["facilities"].map(lambda v: json.loads(v) if isinstance(v, str) else [])
    d["n_facilities"] = fac.map(len)
    
    if fit_mode:
        cnt = collections.Counter(c for l in fac for c in l)
        keep_fac = [c for c, v in cnt.items() if v > 0.02 * len(fac)]  # 2% threshold
        memo["keep_fac"] = keep_fac
    else:
        keep_fac = memo.get("keep_fac", [])
    
    if keep_fac:
        mlb = MultiLabelBinarizer(classes=keep_fac)
        fac_filtered = fac.map(lambda l: [c for c in l if c in set(keep_fac)])
        fac_df = pd.DataFrame(
            mlb.fit_transform(fac_filtered),
            columns=["fac_" + c for c in keep_fac],
            index=d.index
        ).astype("int8")
        d = pd.concat([d, fac_df], axis=1)
    
    # ---- Geographic features (metro-relative) ----
    if fit_mode and all_data is not None:
        metro_centers = all_data.groupby("metro")[["geo_lat", "geo_lon"]].median()
        memo["metro_centers"] = metro_centers
    metro_centers = memo.get("metro_centers")
    
    if metro_centers is not None:
        d["lat_off"] = d["geo_lat"] - d["metro"].map(metro_centers["geo_lat"])
        d["lon_off"] = d["geo_lon"] - d["metro"].map(metro_centers["geo_lon"])
        d["dist_metro_center"] = np.hypot(d["lat_off"], d["lon_off"])
    
    # ---- Group aggregate features (from combined train+test, no target leakage) ----
    if fit_mode and all_data is not None:
        aggs = {}
        aggs["operator_listings"] = all_data["operator_ref"].value_counts()
        aggs["zone_count"] = all_data["zone"].value_counts()
        aggs["zone_mean_seats"] = all_data.groupby("zone")["max_seats"].mean()
        aggs["zone_median_seats"] = all_data.groupby("zone")["max_seats"].median()
        aggs["zone_std_seats"] = all_data.groupby("zone")["max_seats"].std()
        aggs["metro_mean_seats"] = all_data.groupby("metro")["max_seats"].mean()
        aggs["metro_count"] = all_data["metro"].value_counts()
        aggs["borough_count"] = all_data["borough"].value_counts()
        aggs["borough_mean_seats"] = all_data.groupby("borough")["max_seats"].mean()
        aggs["zone_mean_portfolio"] = all_data.groupby("zone")["operator_portfolio_size"].mean()
        aggs["metro_mean_portfolio"] = all_data.groupby("metro")["operator_portfolio_size"].mean()
        memo["aggs"] = aggs
    aggs = memo.get("aggs", {})
    
    if "operator_listings" in aggs:
        d["operator_listings"] = d["operator_ref"].map(aggs["operator_listings"]).fillna(0)
    if "zone_count" in aggs:
        d["zone_count"] = d["zone"].map(aggs["zone_count"]).fillna(0)
    if "zone_mean_seats" in aggs:
        d["zone_mean_seats"] = d["zone"].map(aggs["zone_mean_seats"])
        d["seats_vs_zone"] = d["max_seats"] - d["zone_mean_seats"]
    if "zone_median_seats" in aggs:
        d["zone_median_seats"] = d["zone"].map(aggs["zone_median_seats"])
    if "zone_std_seats" in aggs:
        d["zone_std_seats"] = d["zone"].map(aggs["zone_std_seats"])
    if "metro_mean_seats" in aggs:
        d["metro_mean_seats"] = d["metro"].map(aggs["metro_mean_seats"])
        d["seats_vs_metro"] = d["max_seats"] - d["metro_mean_seats"]
    if "metro_count" in aggs:
        d["metro_count"] = d["metro"].map(aggs["metro_count"]).fillna(0)
    if "borough_count" in aggs:
        d["borough_count"] = d["borough"].map(aggs["borough_count"]).fillna(0)
    if "borough_mean_seats" in aggs:
        d["borough_mean_seats"] = d["borough"].map(aggs["borough_mean_seats"])
        d["seats_vs_borough"] = d["max_seats"] - d["borough_mean_seats"]
    if "zone_mean_portfolio" in aggs:
        d["zone_mean_portfolio"] = d["zone"].map(aggs["zone_mean_portfolio"])
    if "metro_mean_portfolio" in aggs:
        d["metro_mean_portfolio"] = d["metro"].map(aggs["metro_mean_portfolio"])
        d["portfolio_vs_metro"] = d["operator_portfolio_size"] - d["metro_mean_portfolio"]
    
    # ---- Interaction terms ----
    d["seats_x_zones"] = d["max_seats"] * d["zones"]
    d["seats_x_rooms"] = d["max_seats"] * d["utility_rooms"]
    d["score_x_reviews"] = d["score_mean"] * d["review_count"]
    d["tenure_x_portfolio"] = d["platform_tenure"] * d["operator_portfolio_size"]
    d["seats_x_portfolio"] = d["max_seats"] * d["operator_portfolio_size"]
    d["dist_x_seats"] = d.get("dist_metro_center", 0) * d["max_seats"]
    
    # ---- Log transforms of skewed columns ----
    log_cols = ["max_seats", "review_count", "review_count_12m", 
                "operator_portfolio_size", "operator_metro_units",
                "workstations", "utility_rooms"]
    for c in log_cols:
        if c in d.columns:
            d[c + "_log"] = np.log1p(d[c].clip(lower=0))
    
    # ---- Drop raw columns ----
    drop_cols = ["space_id", "operator_ref", "bookable",
                 "snapshot_start_date", "snapshot_end_date",
                 "first_review_date", "last_review_date",
                 "facilities"]
    d = d.drop(columns=[c for c in drop_cols if c in d.columns])
    
    return d, memo

# Build features
all_data = pd.concat([train_clean.drop(columns="price_tier"), test_raw], ignore_index=True)

X_train, memo = build_features(train_clean.drop(columns="price_tier"), all_data=all_data, fit_mode=True)
X_test, _ = build_features(test_raw, all_data=all_data, fit_mode=False, memo=memo)

y = train_clean["price_tier"] - 1  # 0..3
groups = train_clean["operator_ref"].fillna("missing")

# Align columns
common_cols = sorted(set(X_train.columns) & set(X_test.columns))
X_train = X_train[common_cols]
X_test = X_test[common_cols]

print(f"Features: {X_train.shape[1]}")
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")
print(f"Numeric: {len(X_train.select_dtypes('number').columns)}, Cat: {len(X_train.select_dtypes(exclude='number').columns)}")

Features: 226
Train: 278543, Test: 114538
Numeric: 216, Cat: 10


## 3. Prepare Data for Models

In [6]:
cat_cols = X_train.select_dtypes(exclude="number").columns.tolist()
num_cols = X_train.select_dtypes("number").columns.tolist()

print(f"Categorical columns ({len(cat_cols)}): {cat_cols}")
print(f"Numeric columns: {len(num_cols)}")

# For LightGBM/XGBoost: ordinal encode categoricals
oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-1)
X_train_enc = X_train.copy()
X_test_enc = X_test.copy()
X_train_enc[cat_cols] = oe.fit_transform(X_train[cat_cols])
X_test_enc[cat_cols] = oe.transform(X_test[cat_cols])

# For CatBoost: keep categoricals as strings
X_train_cb = X_train.copy()
X_test_cb = X_test.copy()
for c in cat_cols:
    X_train_cb[c] = X_train_cb[c].fillna("_missing_").astype(str)
    X_test_cb[c] = X_test_cb[c].fillna("_missing_").astype(str)

print("\nData prepared for all models.")

Categorical columns (10): ['access_type', 'borough', 'compliance_status', 'metro', 'operator_certified', 'operator_class', 'operator_has_avatar', 'operator_id_verified', 'space_class', 'zone']
Numeric columns: 216

Data prepared for all models.


## 4. Cross-Validation with Multiple Models (OOF Predictions for Stacking)

In [ ]:
N_FOLDS = 5
N_CLASSES = 4
gkf = GroupKFold(n_splits=N_FOLDS)

# Storage for OOF predictions (probabilities)
oof_lgb = np.zeros((len(X_train), N_CLASSES))
oof_xgb = np.zeros((len(X_train), N_CLASSES))
oof_cb = np.zeros((len(X_train), N_CLASSES))

# Storage for test predictions (averaged across folds)
test_lgb = np.zeros((len(X_test), N_CLASSES))
test_xgb = np.zeros((len(X_test), N_CLASSES))
test_cb = np.zeros((len(X_test), N_CLASSES))

fold_scores = {"lgb": [], "xgb": [], "cb": []}

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_train_enc, y, groups)):
    print(f"\n{'='*60}")
    print(f"FOLD {fold+1}/{N_FOLDS} | Train: {len(tr_idx)} | Val: {len(va_idx)}")
    print(f"{'='*60}")
    
    # ---- LightGBM ----
    lgb_params = {
        "objective": "multiclass",
        "num_class": N_CLASSES,
        "metric": "multi_logloss",
        "learning_rate": 0.03,
        "num_leaves": 80,
        "min_child_samples": 100,
        "subsample": 0.8,
        "subsample_freq": 1,
        "colsample_bytree": 0.7,
        "reg_lambda": 2.0,
        "reg_alpha": 0.5,
        "is_unbalance": True,
        "random_state": SEED,
        "n_jobs": -1,
        "verbose": -1,
    }
    dtrain = lgb.Dataset(X_train_enc.iloc[tr_idx], y.iloc[tr_idx])
    dval = lgb.Dataset(X_train_enc.iloc[va_idx], y.iloc[va_idx], reference=dtrain)
    
    lgb_model = lgb.train(
        lgb_params, dtrain,
        num_boost_round=3000,
        valid_sets=[dval],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(500)]
    )
    
    oof_lgb[va_idx] = lgb_model.predict(X_train_enc.iloc[va_idx])
    test_lgb += lgb_model.predict(X_test_enc) / N_FOLDS
    f1_lgb = f1_score(y.iloc[va_idx], oof_lgb[va_idx].argmax(1), average="macro")
    fold_scores["lgb"].append(f1_lgb)
    print(f"  LGB F1: {f1_lgb:.4f} | best_iter: {lgb_model.best_iteration}")
    
    # ---- XGBoost ----
    xgb_model = xgb.XGBClassifier(
        objective="multi:softprob",
        num_class=N_CLASSES,
        n_estimators=3000,
        learning_rate=0.03,
        max_depth=7,
        min_child_weight=50,
        subsample=0.8,
        colsample_bytree=0.7,
        reg_lambda=2.0,
        reg_alpha=0.5,
        tree_method="hist",
        enable_categorical=False,
        random_state=SEED,
        n_jobs=-1,
        verbosity=0,
        early_stopping_rounds=100,
        eval_metric="mlogloss",
    )
    xgb_model.fit(
        X_train_enc.iloc[tr_idx], y.iloc[tr_idx],
        eval_set=[(X_train_enc.iloc[va_idx], y.iloc[va_idx])],
        verbose=False
    )
    
    oof_xgb[va_idx] = xgb_model.predict_proba(X_train_enc.iloc[va_idx])
    test_xgb += xgb_model.predict_proba(X_test_enc) / N_FOLDS
    f1_xgb = f1_score(y.iloc[va_idx], oof_xgb[va_idx].argmax(1), average="macro")
    fold_scores["xgb"].append(f1_xgb)
    print(f"  XGB F1: {f1_xgb:.4f} | best_iter: {xgb_model.best_iteration}")
    
    # ---- CatBoost ----
    cb_model = CatBoostClassifier(
        iterations=3000,
        learning_rate=0.05,
        depth=7,
        l2_leaf_reg=3,
        random_strength=0.5,
        bagging_temperature=0.8,
        border_count=128,
        auto_class_weights="Balanced",
        cat_features=cat_cols,
        random_seed=SEED,
        verbose=0,
        early_stopping_rounds=100,
        eval_metric="TotalF1:average=Macro",
        task_type="CPU",
    )
    cb_model.fit(
        X_train_cb.iloc[tr_idx], y.iloc[tr_idx],
        eval_set=(X_train_cb.iloc[va_idx], y.iloc[va_idx]),
        verbose=0
    )
    
    oof_cb[va_idx] = cb_model.predict_proba(X_train_cb.iloc[va_idx])
    test_cb += cb_model.predict_proba(X_test_cb) / N_FOLDS
    f1_cb = f1_score(y.iloc[va_idx], oof_cb[va_idx].argmax(1), average="macro")
    fold_scores["cb"].append(f1_cb)
    print(f"  CB  F1: {f1_cb:.4f} | best_iter: {cb_model.best_iteration_}")

print("\n" + "="*60)
print("OVERALL CV RESULTS:")
for name in ["lgb", "xgb", "cb"]:
    scores = fold_scores[name]
    print(f"  {name.upper()}: {np.mean(scores):.4f} +/- {np.std(scores):.4f}")


FOLD 1/5 | Train: 222834 | Val: 55709
Training until validation scores don't improve for 100 rounds
[500]	valid_0's multi_logloss: 0.672806


## 5. Stacking / Ensemble

In [ ]:
# Method 1: Simple weighted average (try multiple weight combos)
best_f1 = 0
best_weights = None

for w_lgb in np.arange(0.2, 0.7, 0.05):
    for w_xgb in np.arange(0.1, 0.5, 0.05):
        w_cb = 1.0 - w_lgb - w_xgb
        if w_cb < 0.05 or w_cb > 0.6:
            continue
        blend = w_lgb * oof_lgb + w_xgb * oof_xgb + w_cb * oof_cb
        pred = blend.argmax(1)
        f1 = f1_score(y, pred, average="macro")
        if f1 > best_f1:
            best_f1 = f1
            best_weights = (w_lgb, w_xgb, w_cb)

print(f"Best weighted average: F1={best_f1:.4f}")
print(f"Weights: LGB={best_weights[0]:.2f}, XGB={best_weights[1]:.2f}, CB={best_weights[2]:.2f}")

# Method 2: Stacking with LogisticRegression
stack_features = np.hstack([oof_lgb, oof_xgb, oof_cb])
stack_test = np.hstack([test_lgb, test_xgb, test_cb])

# CV for stacking meta-learner 
stack_oof = np.zeros(len(y))
stack_test_preds = np.zeros((len(X_test), N_CLASSES))

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_train_enc, y, groups)):
    lr = LogisticRegression(max_iter=1000, C=1.0, class_weight="balanced", random_state=SEED)
    lr.fit(stack_features[tr_idx], y.iloc[tr_idx])
    stack_oof[va_idx] = lr.predict(stack_features[va_idx])

f1_stack = f1_score(y, stack_oof, average="macro")
print(f"Stacking (LR meta): F1={f1_stack:.4f}")

# Pick the better method
if f1_stack > best_f1:
    print("\n>> Using STACKING approach")
    lr_final = LogisticRegression(max_iter=1000, C=1.0, class_weight="balanced", random_state=SEED)
    lr_final.fit(stack_features, y)
    final_test_pred = lr_final.predict(stack_test)
    use_stacking = True
else:
    print("\n>> Using WEIGHTED AVERAGE approach")
    final_blend = best_weights[0] * test_lgb + best_weights[1] * test_xgb + best_weights[2] * test_cb
    final_test_pred = final_blend.argmax(1)
    use_stacking = False

## 6. Threshold Optimization for Macro-F1

In [ ]:
# Try optimizing decision thresholds per class to improve macro-F1
# This adjusts the probability thresholds away from argmax

from scipy.optimize import minimize

# Use the best blend probabilities
if use_stacking:
    oof_blend = stack_features  # we'll use raw for threshold opt
    oof_probs = best_weights[0] * oof_lgb + best_weights[1] * oof_xgb + best_weights[2] * oof_cb
else:
    oof_probs = best_weights[0] * oof_lgb + best_weights[1] * oof_xgb + best_weights[2] * oof_cb

def neg_f1_with_bias(biases, probs, true_labels):
    adjusted = probs + biases
    preds = adjusted.argmax(1)
    return -f1_score(true_labels, preds, average="macro")

# Optimize bias terms
init_bias = np.zeros(N_CLASSES)
result = minimize(
    neg_f1_with_bias, init_bias,
    args=(oof_probs, y.values),
    method="Nelder-Mead",
    options={"maxiter": 5000, "xatol": 1e-6, "fatol": 1e-6}
)

optimal_bias = result.x
f1_optimized = -result.fun
f1_plain = f1_score(y, oof_probs.argmax(1), average="macro")

print(f"F1 without bias: {f1_plain:.4f}")
print(f"F1 with optimized bias: {f1_optimized:.4f}")
print(f"Optimal biases: {optimal_bias}")

if f1_optimized > f1_plain + 0.001:  # only use if meaningful gain
    print(">> Using threshold optimization")
    test_blend = best_weights[0] * test_lgb + best_weights[1] * test_xgb + best_weights[2] * test_cb
    final_test_pred = (test_blend + optimal_bias).argmax(1)
else:
    print(">> Skipping threshold optimization (marginal gain)")

## 7. Multi-Seed Ensemble (Extra Robustness)

In [ ]:
# Train additional LightGBM models with different seeds and average
# This reduces variance and often improves leaderboard score by 0.002-0.005

SEEDS = [42, 123, 456, 789, 2024]
multi_seed_test = np.zeros((len(X_test), N_CLASSES))
multi_seed_oof = np.zeros((len(X_train), N_CLASSES))

for seed in SEEDS:
    print(f"\nSeed {seed}:")
    seed_oof = np.zeros((len(X_train), N_CLASSES))
    seed_test = np.zeros((len(X_test), N_CLASSES))
    seed_scores = []
    
    for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_train_enc, y, groups)):
        lgb_params_seed = {
            "objective": "multiclass",
            "num_class": N_CLASSES,
            "metric": "multi_logloss",
            "learning_rate": 0.025,
            "num_leaves": 80,
            "min_child_samples": 100,
            "subsample": 0.8,
            "subsample_freq": 1,
            "colsample_bytree": 0.7,
            "reg_lambda": 2.0,
            "reg_alpha": 0.5,
            "is_unbalance": True,
            "random_state": seed,
            "n_jobs": -1,
            "verbose": -1,
            "feature_fraction_seed": seed,
            "bagging_seed": seed,
            "data_random_seed": seed,
        }
        dtrain = lgb.Dataset(X_train_enc.iloc[tr_idx], y.iloc[tr_idx])
        dval = lgb.Dataset(X_train_enc.iloc[va_idx], y.iloc[va_idx], reference=dtrain)
        
        m = lgb.train(
            lgb_params_seed, dtrain,
            num_boost_round=3000,
            valid_sets=[dval],
            callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)]
        )
        seed_oof[va_idx] = m.predict(X_train_enc.iloc[va_idx])
        seed_test += m.predict(X_test_enc) / N_FOLDS
        f1_fold = f1_score(y.iloc[va_idx], seed_oof[va_idx].argmax(1), average="macro")
        seed_scores.append(f1_fold)
    
    print(f"  Mean F1: {np.mean(seed_scores):.4f}")
    multi_seed_oof += seed_oof / len(SEEDS)
    multi_seed_test += seed_test / len(SEEDS)

# Check if multi-seed averaging helps
f1_multi = f1_score(y, multi_seed_oof.argmax(1), average="macro")
print(f"\nMulti-seed LGB OOF F1: {f1_multi:.4f}")

## 8. Final Ensemble: Combine Everything

In [ ]:
# Try various combinations to find the best
print("Individual model OOF F1:")
print(f"  LGB (single seed):  {f1_score(y, oof_lgb.argmax(1), average='macro'):.4f}")
print(f"  XGB:                {f1_score(y, oof_xgb.argmax(1), average='macro'):.4f}")
print(f"  CatBoost:           {f1_score(y, oof_cb.argmax(1), average='macro'):.4f}")
print(f"  LGB (multi-seed):   {f1_multi:.4f}")

# Now search for best weights including multi-seed LGB
best_final_f1 = 0
best_final_weights = None

for w_lgb_ms in np.arange(0.2, 0.6, 0.05):
    for w_xgb in np.arange(0.1, 0.45, 0.05):
        for w_cb in np.arange(0.1, 0.45, 0.05):
            w_lgb_ss = 1.0 - w_lgb_ms - w_xgb - w_cb
            if w_lgb_ss < 0 or w_lgb_ss > 0.4:
                continue
            blend = w_lgb_ms * multi_seed_oof + w_xgb * oof_xgb + w_cb * oof_cb + w_lgb_ss * oof_lgb
            pred = blend.argmax(1)
            f1 = f1_score(y, pred, average="macro")
            if f1 > best_final_f1:
                best_final_f1 = f1
                best_final_weights = (w_lgb_ms, w_xgb, w_cb, w_lgb_ss)

print(f"\nBest final blend: F1={best_final_f1:.4f}")
print(f"Weights: LGB_MS={best_final_weights[0]:.2f}, XGB={best_final_weights[1]:.2f}, "
      f"CB={best_final_weights[2]:.2f}, LGB_SS={best_final_weights[3]:.2f}")

# Apply to test
final_blend_test = (best_final_weights[0] * multi_seed_test + 
                    best_final_weights[1] * test_xgb + 
                    best_final_weights[2] * test_cb + 
                    best_final_weights[3] * test_lgb)

# Apply threshold optimization on the best blend
best_oof_blend = (best_final_weights[0] * multi_seed_oof + 
                  best_final_weights[1] * oof_xgb + 
                  best_final_weights[2] * oof_cb + 
                  best_final_weights[3] * oof_lgb)

result2 = minimize(
    neg_f1_with_bias, np.zeros(N_CLASSES),
    args=(best_oof_blend, y.values),
    method="Nelder-Mead",
    options={"maxiter": 5000, "xatol": 1e-6, "fatol": 1e-6}
)

f1_opt2 = -result2.fun
print(f"After threshold optimization: F1={f1_opt2:.4f}")

if f1_opt2 > best_final_f1 + 0.0005:
    print(">> Using threshold-optimized predictions")
    final_pred = (final_blend_test + result2.x).argmax(1)
    best_cv_f1 = f1_opt2
else:
    print(">> Using raw argmax predictions")
    final_pred = final_blend_test.argmax(1)
    best_cv_f1 = best_final_f1

## 9. Generate Submission

In [ ]:
id_col, tgt_col = sample_sub.columns[:2]

submission = pd.DataFrame({
    id_col: test_raw["space_id"].values,
    tgt_col: final_pred + 1  # shift 0..3 back to 1..4
})

assert len(submission) == len(sample_sub)
assert set(submission[id_col]) == set(sample_sub[id_col])

submission.to_csv("/kaggle/working/submission.csv", index=False)

print(f"\nSubmission shape: {submission.shape}")
print(f"Tier distribution:\n{submission[tgt_col].value_counts(normalize=True).sort_index().round(3)}")
print(f"\nBest CV macro-F1: {best_cv_f1:.4f}")
print("\nSubmission saved!")

## 10. Also Save Individual Model Submissions (Backup)

In [ ]:
# Save individual model submissions as backups
for name, test_probs in [("lgb", test_lgb), ("xgb", test_xgb), ("cb", test_cb), ("lgb_ms", multi_seed_test)]:
    sub = pd.DataFrame({
        id_col: test_raw["space_id"].values,
        tgt_col: test_probs.argmax(1) + 1
    })
    sub.to_csv(f"/kaggle/working/submission_{name}.csv", index=False)
    oof_f1 = f1_score(y, eval(f"oof_{name.replace('_ms', '')}").argmax(1) if name != "lgb_ms" else multi_seed_oof.argmax(1), average="macro")
    print(f"  {name}: OOF F1={oof_f1:.4f} | tier dist: {sub[tgt_col].value_counts(normalize=True).sort_index().round(3).to_dict()}")

print("\nAll submissions saved. Upload the one with the best CV score.")
print("If ensemble doesn't improve LB, try submitting individual models too.")